# AI Fake News Detection - Classical ML + NLP End-to-End Walkthrough

This notebook demonstrates a complete, step-by-step **Classical Machine Learning and Natural Language Processing (NLP)** pipeline for full-article Fake News Detection using the **WELFake Dataset**.

---

## 1. Introduction & Project Objective

The primary objective of this notebook is to classify news articles as **Fake (0)** or **Real (1)** using statistical and linguistic feature representation.

### Pipeline Stages:
1. **Dataset Loading & Schema Verification** (WELFake Dataset)
2. **Full Article Text Construction** (`title` + `text`)
3. **NLP Preprocessing** (Lowercasing, Regex Cleaning, Stopwords Removal, Porter Stemming)
4. **Stratified Train/Test Split**
5. **TF-IDF Feature Extraction** (Fitted exclusively on `X_train`)
6. **Classifier Training** (Naive Bayes, Logistic Regression, Linear SVM)
7. **5-Fold Cross Validation & GridSearchCV Hyperparameter Tuning**
8. **Evaluation Metrics & Model Comparison**

## 2. Load Dataset & Exploratory Data Analysis

In [ ]:
import os
import pandas as pd
import numpy as np

# Check dataset location (data/news_dataset.csv or data/WELFake_Dataset.csv)
possible_paths = [
    os.path.join("..", "data", "news_dataset.csv"),
    os.path.join("..", "data", "WELFake_Dataset.csv"),
    os.path.join("data", "news_dataset.csv"),
    os.path.join("data", "WELFake_Dataset.csv")
]

dataset_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if not dataset_path:
    raise FileNotFoundError(
        "Dataset not found in data/ directory. "
        "Please place the WELFake dataset (news_dataset.csv or WELFake_Dataset.csv) in the 'data/' directory before executing this notebook."
    )

df = pd.read_csv(dataset_path)
print("[*] Loaded Dataset from:", dataset_path)
print("Dataset Shape:", df.shape)
print("Columns      :", list(df.columns))
df.head()

## 3. Data Cleaning & Full Article Text Assembly
WELFake Dataset Schema:
- `title`: News headline
- `text`: Body content
- `label`: Binary classification target (`0 = Fake`, `1 = Real`)

We concatenate `title` and `text` to form `full_text` for full-article classification.

In [ ]:
if "title" in df.columns and "text" in df.columns:
    df["full_text"] = df["title"].fillna("") + " " + df["text"].fillna("")
elif "text" in df.columns:
    df["full_text"] = df["text"].fillna("")
else:
    raise KeyError("Dataset must contain 'text' or 'title' columns.")

df = df.dropna(subset=["full_text", "label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

print("Processed Articles Count:", len(df))
print("Label Distribution:")
print(df["label"].value_counts())

## 4. NLP Preprocessing
We instantiate `TextPreprocessor(use_stemming=True)` to execute:
- Lowercasing
- URL and HTML stripping
- Non-alphabet regex cleaning
- NLTK English stopword filtering
- NLTK Porter Stemming

In [ ]:
import sys
sys.path.append(".."")
from src.preprocessing import TextPreprocessor

preprocessor = TextPreprocessor(use_stemming=True)
sample = df["full_text"].iloc[0]
print("Raw Article Text  :", sample[:120])
print("Cleaned & Stemmed :", preprocessor.clean_text(sample)[:120])

df["clean_text"] = preprocessor.preprocess_corpus(df["full_text"].tolist())

## 5. Stratified Train/Test Split
Crucial ML Principle: Split dataset into 80% train and 20% test sets **BEFORE** fitting the TF-IDF Vectorizer to ensure 0 data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"Training Set: {len(X_train)} samples | Test Set: {len(X_test)} samples")

## 6. TF-IDF Feature Vectorization
- `fit_transform` on `X_train` strictly
- `transform` on `X_test` using fitted vectorizer vocabulary

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("X_train_tfidf Matrix Shape:", X_train_tfidf.shape)
print("X_test_tfidf Matrix Shape :", X_test_tfidf.shape)

## 7. Model Training & 8. Stratified Cross-Validation
Evaluate Naive Bayes, Logistic Regression, and Linear SVM classifiers using 5-Fold Stratified Cross Validation.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import cross_val_score, GridSearchCV

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Linear SVM": CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, dual="auto"))
}

for name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, y_train, cv=5, scoring="accuracy")
    model.fit(X_train_tfidf, y_train)
    print(f"{name:20s} - 5-Fold CV Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)")

# Hyperparameter tuning with GridSearchCV
print("\nRunning GridSearchCV for Logistic Regression...")
param_grid = {"C": [0.1, 1.0, 10.0]}
grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), param_grid, cv=3, scoring="accuracy")
grid.fit(X_train_tfidf, y_train)
print("Best Params:", grid.best_params_)
print(f"Best CV Score: {grid.best_score_*100:.2f}%")

# Set best tuned estimator as final Logistic Regression model
models["Logistic Regression"] = grid.best_estimator_

## 9. Evaluation & 10. Model Comparison
Compute Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and Classification Report on `X_test_tfidf`.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

results = []
for name, model in models.items():
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

df_res = pd.DataFrame(results)
print("=== Model Comparison Summary ===")
display(df_res)

best_model_name = df_res.sort_values(by="F1-Score", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]
print(f"\nClassification Report ({best_model_name}):")
print(classification_report(y_test, best_model.predict(X_test_tfidf), zero_division=0))

## 11. Conclusion
This notebook demonstrates a complete, reproducible Classical Machine Learning & NLP pipeline on the WELFake Dataset.
- Full article text constructed from title + body.
- NLTK lowercasing, cleaning, stopword filtering, and Porter Stemming applied.
- TF-IDF vectorizer fitted exclusively on training set to prevent data leakage.
- Naive Bayes, Logistic Regression (tuned via GridSearchCV), and Linear SVM evaluated on holdout test data.